# Preprocessing

### Import Library

In [11]:
import pandas as pd
import re
import requests
import torch
import nltk
import sys
import time
import ipywidgets
from tqdm.notebook import tqdm


from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

### Membaca Dataset


In [12]:
df = pd.read_csv(
    "data/dataset_200.csv"
)

print("Jumlah data:", len(df))
print(df.head())

Jumlah data: 200
   id                                         isi_berita  label
0   1  Jakarta - Sebanyak 432 atlet membela Indonesia...  sport
1   2  FIA World Endurance Championship 2026 memasuki...  sport
2   3  Campus League Badminton Regional Semarang tela...  sport
3   4  Kontingen Indonesia bersiap menatap Asian Game...  sport
4   5  Jelang Asian Games 2026, Timnas basket 3x3 put...  sport


In [13]:
print(df.columns)

Index(['id', 'isi_berita', 'label'], dtype='str')


In [14]:
df["isi_berita"]

0      Jakarta - Sebanyak 432 atlet membela Indonesia...
1      FIA World Endurance Championship 2026 memasuki...
2      Campus League Badminton Regional Semarang tela...
3      Kontingen Indonesia bersiap menatap Asian Game...
4      Jelang Asian Games 2026, Timnas basket 3x3 put...
                             ...                        
195    PT TBS Energi Utama Tbk (TOBA) menjadi perusah...
196    Seluruh bandara yang sebelumnya terdampak erup...
197    Kepala Eksekutif Pengawas Bursa Mineral dan Ko...
198    Asosiasi Pengusaha Truk Indonesia (Aptrindo) m...
199    PT Elnusa Petrofin (EPN), anak usaha PT Elnusa...
Name: isi_berita, Length: 200, dtype: str

### Stemming Menggunakan Sastrawi

In [15]:
factory = StemmerFactory()

stemmer = factory.create_stemmer()

### Membuat Fungsi Stemming

In [16]:
def stemming_sastrawi(teks):
    teks = str(teks)

    hasil = stemmer.stem(teks)

    return hasil

### Menerapkan Stemming

In [17]:
total_data = len(df)
hasil_stemming = []

print("Memulai proses stemming...")

for i, teks in enumerate(df["isi_berita"], start=1):

    hasil = stemming_sastrawi(teks)

    hasil_stemming.append(hasil)

    persen = (i / total_data) * 100

    sys.stdout.write(
        f"\rProses stemming: {persen:.2f}% "
        f"({i}/{total_data})"
    )

    sys.stdout.flush()

df["teks_stemming"] = hasil_stemming

print("\nStemming selesai.")

Memulai proses stemming...
Proses stemming: 100.00% (200/200)
Stemming selesai.


### Hasil Stemming

In [18]:
print("TEKS ASLI:")
print(df.loc[0, "isi_berita"])

print("\nTEKS SETELAH STEMMING:")
print(df.loc[0, "teks_stemming"])

TEKS ASLI:
Jakarta - Sebanyak 432 atlet membela Indonesia di Asian Games Aichi-Nagoya 2026. NOC mengadakan Tim Indonesia Day untuk membakar semangat kontingen sebelum bertanding.
Foto Sport
Obor Menyala! Indonesia Siap Berlaga di Asian Games 2026
Sabtu, 29 Agu 2026 22:42 WIB
Anda menyukai artikel ini
Artikel disimpan

TEKS SETELAH STEMMING:
jakarta - banyak 432 atlet bela indonesia di asi games aichi-nagoya 2026 noc ada tim indonesia day untuk bakar semangat kontingen belum tanding foto sport obor nyala indonesia siap laga di asi games 2026 sabtu 29 agu 2026 22 42 wib anda suka artikel ini artikel simpan


### Stopword Removal

In [19]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\javier\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Membuat Daftar Stopword Bahasa Indonesia

In [20]:
stop_words = set(
    stopwords.words("indonesian")
)

print(
    "Jumlah stopword:",
    len(stop_words)
)

Jumlah stopword: 757


### Membuat Fungsi Stopword Removal

In [21]:
def hapus_stopword(teks):
    tokens = str(teks).split()

    hasil = []

    for token in tokens:
        if token not in stop_words:
            hasil.append(token)

    return " ".join(hasil)

### Menerapkan Stopword Removal

In [22]:
df["teks_tanpa_stopword"] = df[
    "teks_stemming"
].apply(
    hapus_stopword
)

### Hasil Stopword Removal

In [23]:
print("HASIL STEMMING:")
print(df.loc[0, "teks_stemming"])

print("\nHASIL STOPWORD REMOVAL:")
print(df.loc[0, "teks_tanpa_stopword"])

HASIL STEMMING:
jakarta - banyak 432 atlet bela indonesia di asi games aichi-nagoya 2026 noc ada tim indonesia day untuk bakar semangat kontingen belum tanding foto sport obor nyala indonesia siap laga di asi games 2026 sabtu 29 agu 2026 22 42 wib anda suka artikel ini artikel simpan

HASIL STOPWORD REMOVAL:
jakarta - 432 atlet bela indonesia asi games aichi-nagoya 2026 noc tim indonesia day bakar semangat kontingen tanding foto sport obor nyala indonesia laga asi games 2026 sabtu 29 agu 2026 22 42 wib suka artikel artikel simpan


### Pembersihan Teks

Yang dilakukan:

- **Lowercase**: Mengubah seluruh huruf menjadi huruf kecil.
- **Menghapus angka**: Menghapus seluruh karakter berupa angka.
- **Menghapus tanda baca**: Menghapus tanda baca seperti `.`, `,`, `!`, `?`, dan sebagainya.
- **Menghapus simbol**: Menghapus karakter atau simbol khusus yang tidak diperlukan.
- **Merapikan spasi**: Menghapus spasi berlebih dan merapikan jarak antar kata.

### Membuat Fungsi Cleaning

In [24]:
def pembersihan_teks(teks):
    teks = str(teks)

    # Mengubah huruf menjadi lowercase
    teks = teks.lower()

    # Menghapus angka
    teks = re.sub(
        r"\d+",
        " ",
        teks
    )

    # Menghapus tanda baca dan simbol
    teks = re.sub(
        r"[^a-zA-ZÀ-ÿ\s]",
        " ",
        teks
    )

    # Menghapus spasi berlebih
    teks = re.sub(
        r"\s+",
        " ",
        teks
    ).strip()

    return teks

### Menerapkan Cleaning

In [25]:
df["teks_bersih"] = df[
    "teks_tanpa_stopword"
].apply(
    pembersihan_teks
)

### Hasil Cleaning

In [26]:
print("SEBELUM CLEANING:")
print(df.loc[0, "teks_tanpa_stopword"])

print("\nSETELAH CLEANING:")
print(df.loc[0, "teks_bersih"])

SEBELUM CLEANING:
jakarta - 432 atlet bela indonesia asi games aichi-nagoya 2026 noc tim indonesia day bakar semangat kontingen tanding foto sport obor nyala indonesia laga asi games 2026 sabtu 29 agu 2026 22 42 wib suka artikel artikel simpan

SETELAH CLEANING:
jakarta atlet bela indonesia asi games aichi nagoya noc tim indonesia day bakar semangat kontingen tanding foto sport obor nyala indonesia laga asi games sabtu agu wib suka artikel artikel simpan


In [28]:
df["teks_bersih"] = df[
    "teks_tanpa_stopword"
].apply(
    pembersihan_teks
)

In [29]:
df.to_csv(
    "data/checkpoint_cleaning.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Checkpoint pembersihan teks berhasil disimpan.")

Checkpoint pembersihan teks berhasil disimpan.


In [1]:
import pandas as pd

df = pd.read_csv(
    "data/checkpoint_cleaning.csv"
)

print(
    "Jumlah data:",
    len(df)
)

print(
    df.columns.tolist()
)

Jumlah data: 200
['id', 'isi_berita', 'label', 'teks_stemming', 'teks_tanpa_stopword', 'teks_bersih']


In [13]:
import pandas as pd

df = pd.read_csv(
    "data/checkpoint_cleaning.csv"
)

df_teks_bersih = df[["teks_bersih"]]

df_teks_bersih.to_csv(
    "data/teks_bersih.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV teks bersih berhasil dibuat.")
print("Jumlah data:", len(df_teks_bersih))

CSV teks bersih berhasil dibuat.
Jumlah data: 200
